# Recall URL/QR v10: audit, data publik dan persiapan CNN karakter
Mengikuti [protokol v10](ai-agent-docs/recall-cnn-v10-plan.md). **CNN belum dilatih:** data publik tambahan dapat dikurasi, tetapi holdout baru dua kelas dengan periode/bukti memadai belum tersedia. Run All memverifikasi audit, intake, exposure dan arsitektur tanpa mengarang hasil akurasi.

Data/label Indonesia tetap pending; penggunaan publik dan runtime shadow v6 tidak berubah. Notebook ini berhenti pada gate data secara eksplisit, bukan menganggap semua lima langkah selesai.

In [1]:
import json
from datetime import datetime, timezone
import pandas as pd
import torch
from IPython.display import display
from guardian_ml.paths import ROOT
from guardian_ml.metadata import checksum
from guardian_ml.family_audit import code_hash, write_json
from guardian_ml.exposure_v10 import completed_v9_reuse, completed_v10_intake
from guardian_ml.cnn_readiness import require_training_readiness
from guardian_ml.character_cnn import CharacterCNN, tokenize

RUN = ROOT / "ml/runs/notebook-recall-cnn-v10"
REPORTS = RUN / "reports"
NOTEBOOK = ROOT / "guardian_lab_url_qr_recall_cnn_v10.ipynb"
assert completed_v9_reuse(ROOT) is not None
assert completed_v10_intake(ROOT) is not None
v9 = json.loads(
    (ROOT / "ml/runs/notebook-model-comparison-v9/reports/frozen-v9.json").read_text()
)
protected = {
    **v9["protected"],
    **{item["path"]: item["sha256"] for item in v9["models"].values()},
}
assert all(checksum(ROOT / p) == digest for p, digest in protected.items())
print(
    "V10 pre-training preparation; no target visits, label inference or classifier fitting."
)

V10 pre-training preparation; no target visits, label inference or classifier fitting.


## 1. Bedah kesalahan development
Audit menggunakan 84.545 tune URL, classifier frozen v8/v6, serta policy FPR development setara. V6 di sini kontrol threshold pembanding v9; threshold API asli tidak berubah. Cache audit terikat source/data/model hash dan snapshot protokol audit asli. Flag `lexically_plain` hanya deskripsi tidak adanya keyword/brand yang ditentukan, bukan label aman.

In [2]:
audit = json.loads((REPORTS / "development-error-audit.json").read_text())
assert audit["scope"] == "development_tune_only" and not audit["test_used"]
assert (
    checksum(ROOT / "ml/src/guardian_ml/recall_audit.py")
    == audit["audit_source_sha256"]
)
assert checksum(REPORTS / "development-audit-protocol.md") == audit["protocol_sha256"]
assert (
    checksum(ROOT / "ml/runs/notebook-tfidf-logistic-v8/data/development-v8.parquet")
    == audit["development_sha256"]
)
assert checksum(ROOT / audit["cases_path"]) == audit["cases_sha256"]
for entry in audit["models"].values():
    assert checksum(ROOT / entry["path"]) == entry["sha256"]
rows = []
for model, slices in audit["slices"].items():
    for name, item in slices.items():
        if name in [
            "all",
            "private_hosting:yes",
            "homepage:yes",
            "lexically_plain:yes",
        ] or name.startswith("source:"):
            rows.append(
                {
                    "model": model,
                    "slice": name,
                    "rows": item["rows"],
                    "phishing": item["phishing"],
                    "benign": item["benign"],
                    "recall_pct": None
                    if item["recall"] is None
                    else 100 * item["recall"],
                    "FPR_pct": None if item["fpr"] is None else 100 * item["fpr"],
                    **item["confusion_matrix"],
                }
            )
display(pd.DataFrame(rows).round(3))
display(pd.DataFrame(audit["development_dates"]))
print("Error audit replay verified; original labels unchanged.")

,model,slice,rows,phishing,benign,recall_pct,FPR_pct,tn,fp,fn,tp
0,full_v8,all,84545,30884,53661,41.028,0.371,53462,199,18213,12671
1,full_v8,source:deepurlbench,20000,2194,17806,44.348,0.820,17660,146,1221,973
2,full_v8,source:knowphish_tr_op,545,346,199,58.960,0.000,199,0,142,204
3,full_v8,source:mendeley2020,1060,573,487,46.946,0.411,485,2,304,269
4,full_v8,source:phishvn_core,2274,373,1901,38.874,0.263,1896,5,228,145
5,full_v8,source:phiusiil,29318,11060,18258,38.888,0.033,18252,6,6759,4301
6,full_v8,source:phresh,31348,16338,15010,41.492,0.266,14970,40,9559,6779
7,full_v8,private_hosting:yes,8273,7448,825,90.024,5.818,777,48,743,6705
8,full_v8,homepage:yes,48232,18589,29643,38.539,0.368,29534,109,11425,7164
9,full_v8,lexically_plain:yes,77863,26155,51708,35.867,0.325,51540,168,16774,9381


,source,rows,missing,min,max
0,deepurlbench,145000,0,2010-10-05T06:10:56+00:00,2023-03-08T07:41:52+00:00
1,knowphish_tr_op,4341,4341,None,None
2,mendeley2020,6215,6215,None,None
3,phishvn_core,14762,14762,None,None
4,phiusiil,174300,174300,None,None
5,phresh,363285,0,2024-07-02T00:00:00+00:00,2025-09-07T00:00:00+00:00


Error audit replay verified; original labels unchanged.


## 2. Data tambahan publik dan karantina
IOBHunter memberi hard-negative FQDN yang diterima peneliti. Review bertanggal 2019-2024, sehingga hanya root URL historis yang didukung; bukan semua path pada domain dan bukan gold terkini. PhishTank memberi positif komunitas terverifikasi. Kedua tier dipisahkan; tidak ada pelabelan oleh LLM atau kunjungan target.

Semua role ditetapkan dengan hash domain sebelum scoring; data baru belum diprediksi model dan belum masuk fitting. 32 kasus IOB unaccepted dikarantina sebagai hash metadata. Sepuluh sumber primer ditinjau; hasil review tidak dianggap pencarian seluruh internet.

In [3]:
intake = json.loads((REPORTS / "public-intake-manifest.json").read_text())
source_audit = json.loads((REPORTS / "data-source-audit.json").read_text())
addition_path = ROOT / intake["outputs"]["public-additions-v10.parquet"]["path"]
additional = pd.read_parquet(addition_path)
assert additional.digest.is_unique
assert additional.groupby("group").partition.nunique().max() == 1
assert not intake["new_cases_predicted"]
display(
    additional.groupby(["training_source", "partition", "label"])
    .size()
    .rename("rows")
    .reset_index()
)
display(pd.DataFrame(source_audit["sources"])[["name", "decision", "reason"]])
registry = json.loads(
    (ROOT / "ml/data/indonesia-gold/registry/manifest.json").read_text()
)
assert not registry["unregistered_runs"]
print(
    "Exposure registry:",
    registry["unique_url_hashes"],
    "URL hashes;",
    registry["unique_domain_hashes"],
    "domain hashes;",
    len(registry["sources"]),
    "caches",
)

,training_source,partition,label,rows
0,iobhunter_human_fp,calibration,0,446
1,iobhunter_human_fp,fit,0,2257
2,iobhunter_human_fp,reserve,0,913
3,iobhunter_human_fp,threshold,0,425
4,iobhunter_human_fp,tune,0,428
5,phishtank_recent,reserve,1,2
6,phishtank_recent,threshold,1,1


,name,decision,reason
0,IOBHunter NDSS2026,accept_historical_hard_negative_only,"Researcher-accepted FQDN false positives, revi..."
1,PhishTank verified online,accept_community_verified_positive_only,Only 3 novel supported URLs after exact/domain...
2,PhiShark2026,not_accessible,"2026 acquisition but approval, institutional i..."
3,Phishing and Legitimate Websites Dataset v2,reject_as_temporal_human_gold,"Bounded ZIP index inspected (21318 entries, 2 ..."
4,URL-Phish v2,reject_as_new_period,"Published March2026, phishing collected Nov202..."
5,CompPhish v3,reject_as_temporal_phishing_gold,"Published July2026, collection Sep2024-Aug2025..."
6,PhishBD2026 v3,reject_as_verified_temporal_both_class_holdout,Raw pool almost entirely positive; per-provide...
7,PhishVN latest public audit,retain_existing_evidence_not_new_sufficient_ho...,Pinned latest commit 0f6b3f92f11a50a0f8db3e131...
8,PhreshPhish latest,already_observed,Latest commit eca603e0ecd4f698f8d4d226c088659e...
9,WWW2026 IPFS artifacts,not_usable_as_requested_holdout,Public root artifact provides monitoring/downl...


Exposure registry: 1158886 URL hashes; 747365 domain hashes; 17 caches


## 5. Holdout baru harus tersedia sebelum langkah training 3/4
Reserve IOB adalah challenge benign historis; reserve PhishTank positif terbaru terlalu kecil. Keduanya tidak digabung lalu diberi klaim accuracy temporal atau FPR trafik. Primary temporal holdout belum tersedia dan training gate wajib gagal. Bukti per kasus dan tanggal tidak dibuat dari tanggal upload.

In [4]:
reserve = additional[additional.partition.eq("reserve")].copy()
cutoff = pd.Timestamp("2025-09-07T23:59:59Z")
dates = pd.to_datetime(reserve.date, utc=True, errors="coerce")
new_benign = int((reserve.label.eq(0) & dates.gt(cutoff)).sum())
new_phishing = int((reserve.label.eq(1) & dates.gt(cutoff)).sum())
readiness = {
    "development_error_audit_complete": True,
    "intake_integrity_verified": True,
    "new_holdout_frozen_before_training": False,
    "holdout_domains_disjoint": True,
    "holdout_labels_traceable": True,
    "holdout_period_verified": False,
    "new_holdout_benign": new_benign,
    "new_holdout_phishing": new_phishing,
    "historical_benign_reserved": int(reserve.label.eq(0).sum()),
    "development_missing_observation_dates": sum(
        r["missing"] for r in audit["development_dates"]
    ),
}
try:
    require_training_readiness(readiness)
except ValueError as error:
    ready = False
    gate_reason = str(error)
else:
    raise RuntimeError(
        "Data conditions changed: freeze the CNN experiment addendum before fitting."
    )
print("CNN training permitted:", ready)
print("Data gate:", gate_reason)
display(pd.Series(readiness).to_frame("status"))
write_json(
    REPORTS / "training-readiness.json",
    {**readiness, "ready": ready, "reason": gate_reason},
)

CNN training permitted: False
Data gate: cnn_training_gate_failed:new_holdout_frozen_before_training,holdout_period_verified,new_holdout_benign_at_least_400,new_holdout_phishing_at_least_200


,status
development_error_audit_complete,True
intake_integrity_verified,True
new_holdout_frozen_before_training,False
holdout_domains_disjoint,True
holdout_labels_traceable,True
holdout_period_verified,False
new_holdout_benign,0
new_holdout_phishing,2
historical_benign_reserved,913
development_missing_observation_dates,199618


## 3. Persiapan CNN karakter tanpa SVD
Modul CNN kecil, tokenizer deterministik scheme-free, padding mask, head/tail truncation dengan token CUT, dan loader state_dict diuji pada fixture sintetis. **Tidak ada optimizer, epoch dataset, checkpoint terlatih atau accuracy CNN** pada tahap ini. Panjang URL fitting diaudit untuk menetapkan konfigurasi berikutnya; karakter panjang yang tidak tertampung harus dilaporkan.

## 4. Perbandingan pada FPR setara
Perbandingan CNN versus LR dengan data fitting yang sama baru boleh dijalankan sesudah gate holdout terpenuhi. LR v8/v6 tetap baseline; tidak ada penurunan threshold dari benchmark atau promosi runtime.

In [5]:
torch.set_num_threads(4)
with torch.random.fork_rng():
    torch.manual_seed(20261101)
    architecture = CharacterCNN().eval()
    tokens, truncated = tokenize(
        ["http://example.org/page", "https://example.org/page"]
    )
    with torch.no_grad():
        output = architecture(tokens)
    assert torch.equal(tokens[0], tokens[1]) and torch.isfinite(output).all()
    parameter_count = sum(p.numel() for p in architecture.parameters())
parent = pd.read_parquet(
    ROOT / "ml/runs/notebook-tfidf-logistic-v8/data/development-v8.parquet",
    columns=["partition", "core", "label"],
)
fitting = parent[parent.partition.eq("fit")].copy()
lengths = fitting.core.str.replace(r"^https?://", "", regex=True).str.len()
length_audit = {
    "fitting_rows": len(fitting),
    "quantiles": {
        str(q): float(lengths.quantile(q)) for q in [0.5, 0.9, 0.95, 0.99, 0.995, 0.999]
    },
    "max": int(lengths.max()),
    "over_512": int(lengths.gt(512).sum()),
    "over_1024": int(lengths.gt(1024).sum()),
}
write_json(REPORTS / "fitting-length-audit.json", length_audit)
print(
    "Untrained architecture parameters:",
    parameter_count,
    "torch:",
    torch.__version__,
    "CUDA:",
    torch.cuda.is_available(),
)
display(pd.Series(length_audit).to_frame("value"))

Untrained architecture parameters: 18193 torch: 2.11.0+cpu CUDA: False


,value
fitting_rows,535504
quantiles,"{'0.5': 30.0, '0.9': 63.0, '0.95': 77.0, '0.99..."
max,4183
over_512,46
over_1024,14


## Status aktual dan integritas
Audit selesai dan data tambahan dikurasi; CNN baru arsitektur yang diuji secara teknis. Kekurangan holdout dilaporkan agar tidak berubah menjadi klaim training/evaluasi selesai. Tidak membuat reviewer Indonesia atau label manusia baru.

In [6]:
assert all(checksum(ROOT / p) == digest for p, digest in protected.items())
assert not list((RUN / "artifacts").glob("*.pt"))
status = {
    "updated_at_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": NOTEBOOK.name,
    "notebook_code_sha256": code_hash(NOTEBOOK),
    "protocol_sha256": checksum(ROOT / "ai-agent-docs/recall-cnn-v10-plan.md"),
    "audit_sha256": checksum(REPORTS / "development-error-audit.json"),
    "intake_manifest_sha256": checksum(REPORTS / "public-intake-manifest.json"),
    "source_audit_sha256": checksum(REPORTS / "data-source-audit.json"),
    "stages": {
        "development_error_audit": "completed",
        "public_additions": "curated_not_fitted",
        "cnn": "architecture_verified_untrained",
        "comparison": "not_started_data_gate",
        "new_temporal_holdout": "not_available",
    },
    "training_ready": False,
    "training_gate": readiness,
    "new_rows_fitted": 0,
    "cnn_accuracy": None,
    "new_temporal_benign_rows": 0,
    "additional_rows_curated": len(additional),
    "runtime": "original_frozen_v6_unchanged",
    "indonesia_gold_available": False,
    "protected_checksums": protected,
    "production_ready": False,
}
write_json(REPORTS / "status-v10.json", status)
write_json(ROOT / "ml/reports/recall-cnn-v10.json", status)
print(
    "STATUS: awaiting valid new holdout; no CNN fitting or benchmark scoring performed."
)
print(
    "Historical hard negatives reserved:",
    readiness["historical_benign_reserved"],
    "Recent positives reserved:",
    new_phishing,
)

STATUS: awaiting valid new holdout; no CNN fitting or benchmark scoring performed.
Historical hard negatives reserved: 913 Recent positives reserved: 2
